# Nasdaq ITCH Feed Handler v2

This processig system is for the v2 release of the Nasdaq ITCH Feed Handler. In its default state, it tracks the Apple, Microsoft, and Netflix stocks using Nasdaq ITCH historical data from 30/12/2019


In [ ]:
# imports

from pynq import Overlay, allocate, MMIO
import numpy as np
import time
import gzip
import struct
import math
import golden.runner as test
import json
import pandas as pd
import urllib.request
import io
import ssl

### Key Note before use:


For the variables below, you are free to change them, however, there are some notes you should be aware of if you are trying to change these variables

- The `SW_TARGET_SYMBOL` can only be used **once** to track **one stock**. So ensure that you pick one stock that you are also tracking in hardware
- Ensure that the stock you track is actually in the Nasdaq (at the date for the historical data), and for the hardware stock ensure that there are **4 spaces after the last letter of the symbol**, or the system will fail
- Base price values are given for the default stocks, given that wrapping (i.e. values below the base price) are **not supported**, ensure that the base price is **below the minimum price you expect for the histroical data**
- Ensure the data path you use includes the sections `localhost:8443`, or with the relevant port used to open a reverse SSH Tunnel

In [ ]:
PATH        = "/home/xilinx/jupyter_notebooks/Nasdaq/v2release.bit"
DATA_URL    = "https://localhost:8443/ITCH/Nasdaq%20ITCH/12302019.NASDAQ_ITCH50.gz"

SW_MESSAGES_TO_READ = 1_000_000 # number of messages read in software - Recommended to keep below this number - data extraction may take a very long time otherwise

MSG_LIMIT        = True # Hardware will read double the above number of messages also if true


SW_TARGET_SYMBOL = "MSFT" # ensure this matches a Hardware value

HW_SYMBOL_0      = b"AAPL    " # This is 4 spaces after the L - required for accurate data
HW_SYMBOL_1      = b"MSFT    "
HW_SYMBOL_2      = b"NFLX    "

BASE_PRICE_STOCK_0  = 0x5208 # base price of $210.00 (for apple)
BASE_PRICE_STOCK_1  = 0x2EE0 # base price of $120.00 (for microsoft)
BASE_PRICE_STOCK_2  = 0x7D00 # base price of $320.00 (for Netflix)


This function is used to generate network headers to pass to the system, the historical data filters these out, but we include them to allow for stripping of network headers in our system

In [22]:
# network header gen function

def generate_network_headers(payload_len, seq_num):

    mold_len = 20 + 2 + payload_len     
    udp_len = 8 + mold_len              
    ip_total_len = 20 + udp_len          

    # Ethernet - 14 bytes of form: Dest MAC, Src MAC, EtherType (0x0800 for IPv4)
    eth = struct.pack('!6s6sH', b'\x00\x11\x22\x33\x44\x55', b'\xAA\xBB\xCC\xDD\xEE\xFF', 0x0800)

    # 2. IPv4 - 20 bytes of form: VHL, TOS, TotalLen, ID, Flags/Frag, TTL, Protocol (17=UDP), Checksum, SrcIP, DestIP
    ip = struct.pack('!BBHHHBBH4s4s',
                     0x45, 0x00, ip_total_len, 0x0000, 0x0000, 
                     64, 17, 0x0000, 
                     b'\xc0\xa8\x01\x64', b'\xc0\xa8\x01\xc8')

    # 3. UDP - 8 bytes of form: SrcPort, DestPort, Length, Checksum
    udp = struct.pack('!HHHH', 12345, 12345, udp_len, 0x0000)

    # 4. MoldUDP64 - 20 bytes of form: Session (10 bytes), Sequence Number (8 bytes), Message Count (2 bytes)
    mold = struct.pack('!10sQH', b'SESSION123', seq_num, 1)

    # 5. Mold Message Block Length 2 bytes
    msg_len_hdr = struct.pack('!H', payload_len)

    return eth + ip + udp + mold + msg_len_hdr

This function is used to obtain the gzip file from the URL, if successful, there will be a remote connection via a reverse SSH tunnel. Otherwise, the file will be downloaded locally

In [ ]:
def get_gzip_stream(url_or_path):

    if url_or_path.startswith(("http://", "https://")):
        print("Enabling remote connection")
        ssl_ctx = ssl.create_default_context()
        ssl_ctx.check_hostname = False
        ssl_ctx.verify_mode = ssl.CERT_NONE
        req = urllib.request.Request(
            url_or_path,
            headers={
                "Host": "emi.nasdaq.com",
                "User-Agent": "Mozilla/5.0",
            },
        )
        response = urllib.request.urlopen(req, context=ssl_ctx)
        gz_file = gzip.GzipFile(fileobj=response)
        print("Remote connection successful!")
        return io.BufferedReader(gz_file, buffer_size=1024 * 1024), response
    else:
        print("Enabling local download")
        ## Open file locally if remote connection fails
        gz_file = gzip.open(url_or_path, "rb")
        print("Download complete!")
        return io.BufferedReader(gz_file, buffer_size=1024 * 1024), None

Download the overlay for hardware

In [ ]:
# Overlay Load

ol = Overlay(PATH)
ol.download()
print("Overlay Loaded:")
dma = ol.axi_dma_0
gpio_bid   = ol.axi_gpio_0
gpio_ask   = ol.axi_gpio_1
gpio_base_price1 = ol.axi_gpio_3
gpio_base_price2 = ol.axi_gpio_4
gpio_stock_id    = ol.axi_gpio_5

Variables and other declarations for hardware use

In [ ]:
send_buffer = allocate(shape=(256,), dtype=np.uint32)

HW_MESSAGES_TO_READ = 2 * SW_MESSAGES_TO_READ

files = {
    1: open(f"hw_{HW_SYMBOL_0[0:4].decode()}_data.txt", "w"),
    2: open(f"hw_{HW_SYMBOL_1[0:4].decode()}_data.txt", "w"),
    3: open(f"hw_{HW_SYMBOL_2[0:4].decode()}_data.txt", "w")
}

json_files = {
    1: open(f"hw_{HW_SYMBOL_0[0:4].decode()}_data.jsonl", "w"),
    2: open(f"hw_{HW_SYMBOL_1[0:4].decode()}_data.jsonl", "w"),
    3: open(f"hw_{HW_SYMBOL_2[0:4].decode()}_data.jsonl", "w")
}

target_symbols = {
    HW_SYMBOL_0[0:4]: 1,
    HW_SYMBOL_1[0:4]: 2,
    HW_SYMBOL_2[0:4]: 3,
}

target_locate_0   = None
target_locate_1   = None
target_locate_2   = None

gpio_base_price1.channel1.write(BASE_PRICE_STOCK_0, 0xffff_ffff)
gpio_base_price1.channel2.write(BASE_PRICE_STOCK_1, 0xffff_ffff)
gpio_base_price2.channel1.write(BASE_PRICE_STOCK_2, 0xffff_ffff)

# Trackers
msg_count = 0
target_msg_count = 0
last_bid_price = {1: 0, 2: 0, 3: 0}
last_ask_price = {1: 0, 2: 0, 3: 0}
last_bid_shares = {1: 0, 2: 0, 3: 0}
last_ask_shares = {1: 0, 2: 0, 3: 0}
locate_to_stock_id = {}


if not dma.sendchannel.running:
    dma.sendchannel.start()
    

## Hardware run

This wil run the hardware design, outputting data in a hw_data files

The text file is a readable file for the user

The JSONL file is used for the Hardware/Software Comparison


In [ ]:
stream_render, net_response = get_gzip_stream(DATA_URL)

try:
    with stream_render as data:
        while True:
            start_msg = data.read(2) # Reading the 2-byte length header
            if not start_msg: break

            ins_len = int.from_bytes(start_msg, byteorder='big')
            output_data = data.read(ins_len)
            if len(output_data) < ins_len: break

            msg_count += 1
            if (msg_count > HW_MESSAGES_TO_READ) and MSG_LIMIT: 
                print("Message Limit Reached")
                if net_response:
                    net_response.close()
                    print("Remote connection closed")
                for f in files.values():
                    f.close()
                break
            msg_type = output_data[0:1]
            locate_code = output_data[1:3]

            if msg_type == b'R':
                symbol = output_data[11:15]      

                if symbol in target_symbols:
                    s_id = target_symbols[symbol]
                    locate_to_stock_id[locate_code] = s_id
                    print(f"Stock {symbol.decode()} mapped to ID {s_id} (Locate: {locate_code.hex()})")

            # Message modify step for Hardware
            modified_msg = bytearray(output_data)
            keep_message = False

            s_id = locate_to_stock_id.get(locate_code)

            if msg_type == b'S': keep_message = True
            elif s_id is not None:
                modified_msg[1:3] = s_id.to_bytes(2, byteorder='big')

                # Price Division for hardware use
                offset = 32 if msg_type in [b'A', b'F', b'C'] else (31 if msg_type == b'U' else None)
                if offset:
                    price = int.from_bytes(modified_msg[offset:offset+4], 'big') // 100
                    modified_msg[offset:offset+4] = price.to_bytes(4, 'big')
                keep_message = True


            if keep_message:

                # Generate the 64-byte network header chain
                network_headers = generate_network_headers(payload_len=ins_len, seq_num=msg_count)

                full_packet = network_headers + modified_msg

                # Pad total packet to 4-byte boundary
                packet_len = len(full_packet)
                pad_len = math.ceil(packet_len / 4) * 4
                padded = full_packet.ljust(pad_len, b'\x00')

                # Big Endian swap
                temp_arr = np.frombuffer(padded, dtype=np.uint32)
                send_buffer.fill(0)
                send_buffer[:len(temp_arr)] = temp_arr.byteswap()

                dma.sendchannel.transfer(send_buffer[: len(temp_arr)], nbytes=pad_len)
                dma.sendchannel.wait()

                if msg_type != b'S' and s_id in last_bid_price:                              

                    hw_s_id = gpio_stock_id.channel1.read()

                    if hw_s_id in last_bid_price:
                        bid_price = gpio_bid.channel1.read()
                        ask_price = gpio_ask.channel1.read()
                        bid_shares = gpio_bid.channel2.read()
                        ask_shares = gpio_ask.channel2.read()

                        # Capture updates even if only the shares change
                        if (bid_price != last_bid_price[hw_s_id] or 
                            ask_price != last_ask_price[hw_s_id] or
                            bid_shares != last_bid_shares[hw_s_id] or
                            ask_shares != last_ask_shares[hw_s_id]):

                            last_bid_price[hw_s_id], last_ask_price[hw_s_id] = bid_price, ask_price
                            last_bid_shares[hw_s_id], last_ask_shares[hw_s_id] = bid_shares, ask_shares

                            bid_formatted = f"${(last_bid_price[hw_s_id] / 100.0):.2f}"
                            ask_formatted = f"${(last_ask_price[hw_s_id] / 100.0):.2f}"

                            files[hw_s_id].write(
                                f"{last_bid_shares[hw_s_id]:>4} Bid shares at price: {bid_formatted:>8} | "
                                f"{last_ask_shares[hw_s_id]:>4} Ask shares at price: {ask_formatted:>8}\n"
                            )


                            bbo_data = {"bbo": {"ask_price": last_ask_price[hw_s_id]*100, "ask_size": last_ask_shares[hw_s_id], 
                                                "bid_price": last_bid_price[hw_s_id]*100, "bid_size": last_bid_shares[hw_s_id]}}

                            json_data = json.dumps(bbo_data) 
                            json_files[hw_s_id].write(json_data + "\n")

finally:
    print("All data obtained")
    if net_response:
        net_response.close()
        print("Remote connection closed")
    for f in files.values():
        f.close()

Stock AAPL mapped to ID 1 (Locate: 000d)
Stock MSFT mapped to ID 2 (Locate: 14ab)
Stock NFLX mapped to ID 3 (Locate: 1579)


## Software run

This run uses the golden model with the same historical data, we only read 1,000,000 of the messages (this can be altered). Relevent data in JSON form is in the file `golden_states.jsonl`

In [ ]:
# Test with golden model

subsection = bytearray()

stream_render, net_response = get_gzip_stream(DATA_URL)

with stream_render as data:
    for msg in range(SW_MESSAGES_TO_READ):
        start_msg = data.read(2) # Reading the 2-byte length header
        if not start_msg: 
            break
            
        ins_len = int.from_bytes(start_msg, byteorder='big')
        output_data = data.read(ins_len)
        if len(output_data) < ins_len: 
            break
        
        subsection.extend(start_msg)
        subsection.extend(output_data)

if net_response:
        net_response.close()
        print("Remote connection closed")
print(f"Extracted {SW_MESSAGES_TO_READ} messages. Running golden model...")

with open(f"golden_events_{SW_TARGET_SYMBOL}.jsonl", "w", encoding="utf-8") as events_out, \
     open(f"golden_states_{SW_TARGET_SYMBOL}.jsonl", "w", encoding="utf-8") as states_out:
    
    # run_bytes natively accepts bytearray objects
    test.run_bytes(
        data=subsection,
        events_out=events_out,
        states_out=states_out,
        symbol=SW_TARGET_SYMBOL
    )

print("Subsection processing complete!")

Extracted 1000000 messages. Running golden model...
Subsection processing complete!


## Hardware/Software Comparison

This cell compares the software data and hardware data. Any errors will appear in the `error_log.txt` file

In [ ]:
# Data comparisons

HW_TARGET_FILE = f"hw_{SW_TARGET_SYMBOL}_data.jsonl"
error          = open("error_log.txt", "w")

last_sw_ask_price  = None
last_sw_bid_price  = None
last_sw_ask_shares = None
last_sw_bid_shares = None

num_errors = 0
hw_idx     = 0

## Software data in JSON file read

software_data = pd.read_json(f"golden_states_{SW_TARGET_SYMBOL}.jsonl", lines=True)

    
## Hardware data in txt file read

hardware_data = pd.read_json(HW_TARGET_FILE, lines=True) 

## Comparison
for bbo in software_data["bbo"]:
    sw_ask_price  = bbo["ask_price"]
    sw_bid_price  = bbo["bid_price"]
    sw_ask_shares = bbo["ask_size"]
    sw_bid_shares = bbo["bid_size"]
    
    if(sw_ask_price  != last_sw_ask_price  or
       sw_bid_price  != last_sw_bid_price  or
       sw_ask_shares != last_sw_ask_shares or
       sw_bid_shares != last_sw_bid_shares ):
        
        last_sw_ask_price,  last_sw_bid_price  = sw_ask_price,  sw_bid_price
        last_sw_ask_shares, last_sw_bid_shares = sw_ask_shares, sw_bid_shares
        
        hw_bbo = hardware_data["bbo"].iloc[hw_idx]
        
        hw_ask_price  = hw_bbo["ask_price"]
        hw_bid_price  = hw_bbo["bid_price"]
        hw_ask_shares = hw_bbo["ask_size"]
        hw_bid_shares = hw_bbo["bid_size"]
        
        
        if(last_sw_ask_price  !=  hw_ask_price  or
           last_sw_bid_price  !=  hw_bid_price  or
           last_sw_ask_shares !=  hw_ask_shares or
           last_sw_bid_shares !=  hw_bid_shares ):
            
            error.write(f"Expected Values: {bbo}, Hardware Values: {hardware_data['bbo'].iloc[hw_idx]}\n")
            
        hw_idx += 1
    
with open("error_log.txt", "r") as err:
    num_errors = sum(1 for line in err)


print("Comparison Complete!\n")
print(f"Number of errors: {num_errors}")
